# Gjenkjenn klær med et nevralt nett

**Du skal kunne:** følge et bilde fra piksler til logits, cross entropy og
vektoppdatering. Vi bruker Fashion-MNIST: små gråtonebilder av ti typer klær.
Det er en variant av det klassiske MNIST-eksempelet med håndskrevne sifre.
Denne øvelsen kan kjøres på laptop. Første kjøring laster ned datasettet.


## Gjør arbeidsbenken klar

`import` henter verktøy; `as` gir et kortere navn. `Path` håndterer filstier.
Vi finner kursmappen og legger `startkode` i Pythons søkesti, slik at samme kode
virker fra VS Code og JupyterLab. `sett_seed(42)` gir et fast startpunkt for
slumptall; det gjør sammenligning lettere, men lover ikke identiske tall på ulike
maskiner. `velg_enhet()` velger NVIDIA-GPU når CUDA er tilgjengelig, ellers CPU.
En **enhet** er stedet beregningene skjer. Modell og data må ligge på samme enhet.

**Før kjøring:** Gjett om utskriften blir `cpu` eller `cuda`.
**Etter kjøring:** Kontroller at det stemmer med installasjonssjekken.


In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

# Finn kursmappen både fra en notatbok og fra prosjektroten.
her = Path.cwd().resolve()
kandidater = [her, *her.parents, her / "05-dypere-maskinlaering"]
kursmappe = next(p for p in kandidater if (p / "startkode" / "kursverktoy.py").exists())
if str(kursmappe / "startkode") not in sys.path:
    sys.path.insert(0, str(kursmappe / "startkode"))
from kursverktoy import sett_seed, velg_enhet
sett_seed(42)
enhet = velg_enhet()
print("PyTorch:", torch.__version__, "| beregner på:", enhet)


## Del opp øvingsark og prøveark

Vi deler de offisielle treningsdataene i trening og validering med fast seed.
**Trening** er øvingsarkene med fasit. **Validering** er prøveeksamen som hjelper oss
å velge innstillinger. **Test** er slutteksamen vi sparer til valgene er tatt.
Vi laster ikke inn testsettet her.

`ToTensor()` konverterer bildet til desimaltall mellom 0 og 1 med form `[1, 28, 28]`.
`random_split` fordeler indeksene tilfeldig uten overlapp. `Subset` gir et lite
startutvalg. En `DataLoader` leverer pakker med 64 bilder; `shuffle=True` stokker
treningsrekkefølgen. Fasit er heltall fra 0 til 9, ikke sannsynligheter.

**Før kjøring:** Vi har 60 000 offisielle treningsbilder. Hvor mange er igjen til
validering etter at 50 000 er satt av til trening?


In [ ]:
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms

transformasjon = transforms.ToTensor()
alle = datasets.FashionMNIST(kursmappe / "data", train=True,
                            download=True, transform=transformasjon)
trening, validering = random_split(
    alle, [50000, 10000], generator=torch.Generator().manual_seed(42))
# Start med et mindre utvalg. Sett til 50000 og 10000 når alt fungerer.
antall_trening, antall_validering = 6000, 1000
trening = Subset(trening, range(antall_trening))
validering = Subset(validering, range(antall_validering))
trening_laster = DataLoader(trening, batch_size=64, shuffle=True, num_workers=0)
validering_laster = DataLoader(validering, batch_size=64, shuffle=False, num_workers=0)
klasser = ["T-skjorte", "Bukse", "Genser", "Kjole", "Kåpe",
           "Sandal", "Skjorte", "Joggesko", "Veske", "Støvlett"]
print(len(trening), "treningsbilder og", len(validering), "valideringsbilder")


## Se det modellen får se

`subplots` lager et rutenett. `zip` går gjennom akser og bilder parvis, `imshow`
tegner pikslene og `set_title` skriver kategorien. Indeks 0 fjerner kanalaksen
når vi viser et gråtonebilde. `.item()` henter kategorinummeret som Python-tall.

**Stopp før trening:** Kan du selv skille skjorte fra T-skjorte i alle bildene?


In [ ]:
bilder, fasit = next(iter(trening_laster))
print("Bilder:", bilder.shape, "fasit:", fasit.shape, fasit.dtype)
fig, akser = plt.subplots(2, 4, figsize=(9, 5))
for akse, bilde, kategori in zip(akser.flat, bilder[:8], fasit[:8]):
    akse.imshow(bilde[0], cmap="gray")
    akse.set_title(klasser[kategori.item()])
    akse.axis("off")
plt.tight_layout()
plt.show()


## Fra piksler til ti poengsummer

`Sequential` kjeder lag sammen. `Flatten` gir 784 piksler per bilde;
`Linear(784, 128)` lærer 128 kombinasjoner av disse tallene, `ReLU` setter negative
resultater til null, og `Linear(128, 10)` lager ti **logits**, én rå poengsum per klasse.
Uten ReLU kunne disse to lineære lagene slås sammen til ett lineært lag.

`CrossEntropyLoss` tar logits og heltallsfasit. Ikke legg softmax i modellen før
feilmålet; PyTorch håndterer log-softmax inni beregningen på en numerisk stabil måte.
Til visning kan vi bruke `softmax`: da summerer de ti tallene til 1. Det gjør dem
ikke automatisk til pålitelige mål på usikkerhet.

**Forutsi:** Hvilken form skal modellen gi for 64 bilder?


In [ ]:
from torch import nn
sett_seed(42)
modell = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10)).to(enhet)
feilmaal = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(modell.parameters(), lr=0.001)
print("Output:", modell(bilder.to(enhet)).shape)
print("Lærbare parametre:", sum(p.numel() for p in modell.parameters()))


## Hvorfor et skråsikkert feil svar straffes mer

To tenkte modeller skal velge mellom to kategorier. Fasit er kategori 0. Begge
gjetter feil, men den første gir fasit sannsynlighet 0,4 og den andre bare 0,001.
`log()` gjør sannsynlighetene til et mulig sett logits. Dette er bare et regneeksempel,
ikke et ekstra lag vi skal legge inn i modellen. `reduction="none"` gir ett tap per
eksempel i stedet for gjennomsnittet.

**Forutsi:** Hvilken av de to får størst tap, selv om begge har 0 % treff her?


In [ ]:
to_modeller = torch.tensor([[0.4, 0.6], [0.001, 0.999]])
fasit_eksempel = torch.tensor([0, 0])
tap = nn.functional.cross_entropy(to_modeller.log(), fasit_eksempel, reduction="none")
print("Tap for begge:", tap)
print("Samme som -log(p for fasit):", -to_modeller[:, 0].log())


**Kontroll:** Tapene er omtrent 0,916 og 6,908. Treffandel alene skiller ikke
mellom disse feilene, men cross entropy straffer det skråsikre feilsporet mer.


## Les hele treningsløkken

`train()` setter treningsmodus; det får betydning for dropout og BatchNorm senere.
Hver indre runde er ett **minibatch-steg**, mens en **epoke** er en gjennomgang av
alle treningsbildene vi har valgt. Adam bruker gradienter og en løpende historikk
for å tilpasse oppdateringen per parameter.

Følg de seks merkede linjene. Vi ganger batchens gjennomsnittsfeil med antall bilder
før summering, slik at en liten siste batch ikke teller like mye som en full batch.
`argmax(1)` velger klassen med høyest logit for hvert bilde.

Ved validering brukes både `eval()` og `no_grad()`: det første endrer lagmodus,
det andre slår av sporing av gradienter. Vi kaller aldri `optimizer.step()` der.


In [ ]:
epoker = 3
historikk = []
for epoke in range(epoker):
    modell.train()
    sum_feil, antall = 0.0, 0
    for bilder, fasit in trening_laster:
        bilder, fasit = bilder.to(enhet), fasit.to(enhet)  # 1: flytt pakken
        optimizer.zero_grad()                           # 2: nullstill
        logits = modell(bilder)                         # 3: gjett
        feil = feilmaal(logits, fasit)                   # 4: mål feil
        feil.backward()                                 # 5: regn gradienter
        optimizer.step()                                # 6: juster vekter
        sum_feil += feil.item() * len(fasit)
        antall += len(fasit)
    modell.eval()
    val_sum, riktige, val_antall = 0.0, 0, 0
    with torch.no_grad():
        for bilder, fasit in validering_laster:
            bilder, fasit = bilder.to(enhet), fasit.to(enhet)
            logits = modell(bilder)
            val_sum += feilmaal(logits, fasit).item() * len(fasit)
            riktige += (logits.argmax(1) == fasit).sum().item()
            val_antall += len(fasit)
    historikk.append((sum_feil / antall, val_sum / val_antall))
    print(f"Epoke {epoke+1}: trening={sum_feil/antall:.3f}, "
          f"validering={val_sum/val_antall:.3f}, treff={riktige/val_antall:.1%}")


## Tegn læringen og undersøk feil

Første graf sammenligner feilen på trening og validering. Synkende treningsfeil
viser tilpasning til øvingsarkene; synkende valideringsfeil tyder på bedre
generalisering til samme type nye data. Det er ikke garantert at kurvene synker
hver epoke. `cpu()` henter tallene tilbake slik at Matplotlib kan tegne dem.

Bildene nedenfor kommer fra **validering**, slik at vi fortsatt sparer testsettet.
Vi viser modellens kategori og fasit sammen; fargen gjør feil lette å oppdage.


In [ ]:
plt.plot(range(1, epoker+1), [h[0] for h in historikk], label="Trening")
plt.plot(range(1, epoker+1), [h[1] for h in historikk], label="Validering")
plt.xlabel("Epoke")
plt.ylabel("Cross entropy")
plt.legend()
plt.show()
bilder, fasit = next(iter(validering_laster))
modell.eval()
with torch.no_grad():
    sannsynlighet = modell(bilder.to(enhet)).softmax(1).cpu()
gjett = sannsynlighet.argmax(1)
fig, akser = plt.subplots(2, 4, figsize=(12, 5))
for i, akse in enumerate(akser.flat):
    akse.imshow(bilder[i, 0], cmap="gray")
    akse.set_title(f"Gjett: {klasser[gjett[i]]}\nFasit: {klasser[fasit[i]]}",
                   color="green" if gjett[i] == fasit[i] else "red")
    akse.axis("off")
plt.tight_layout()
plt.show()


## Eksperiment og egen forklaring

Endre bare skjult lag fra 128 til 32. Restart og kjør alt, slik at både modell og
optimizer blir nye. Noter antall parametre og valideringstreff. Unngå å velge
modell ved å prøve stadig flere innstillinger mot sluttesten.

**Kontroll:** Input har form `[64, 1, 28, 28]`, output `[64, 10]` og fasit `[64]`.
Et helt tilfeldig, uniformt tips gir forventet treffandel 10 % og cross entropy
omtrent `-log(0.1) = 2.30`. En utrent modell gir ikke nødvendigvis akkurat disse tallene.

**Forklar for sidemannen:** Hvorfor er `backward()` alene utilstrekkelig for læring?

Kildene til API-ene og datasettet finnes i [kildelisten](../KILDER.md).
**Neste:** [03-overtilpasning-og-eksperimenter.ipynb](03-overtilpasning-og-eksperimenter.ipynb).
